# Incremental Load Control Testing

#### This notebook validates batch registration, pipeline state changes and watermark updates before the same logic is moved into production Python.

In [ ]:
import sys
from pathlib import Path


# The notebook runs from the /notebooks directory.
# Move one level up to reach the project root.
project_root = Path.cwd().parent


# Convert the project root to text because sys.path stores strings.
project_root_str = str(project_root)


# Add the project root only if Python does not already know about it.
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)


print("Project root:", project_root)


# Confirm Python can now locate the src package.
print("src available:", (project_root / "src").exists())

Project root: /Users/mac/Documents/netflix-data-engineering
src available: True


In [2]:
# Import os so we can inspect the environment variables currently loaded.
import os

# Import the dotenv loader.
from dotenv import load_dotenv

# Reload variables from the project .env file.
load_dotenv(override=True)

# Display only non-secret connection settings.
print("Host:", os.getenv("NETFLIX_DB_HOST"))
print("Port:", os.getenv("NETFLIX_DB_PORT"))
print("Database:", os.getenv("NETFLIX_DB_NAME"))
print("ETL user:", os.getenv("NETFLIX_ETL_USER"))

Host: 127.0.0.1
Port: 5432
Database: postgres
ETL user: netflix_etl


## Connect as the ETL user

Confirm that the notebook connects to PostgreSQL using the dedicated `netflix_etl` account before manipulating pipeline metadata.

In [ ]:
import os

# Import psycopg for PostgreSQL connections.
import psycopg

# Import load_dotenv so variables can be loaded from .env.
from dotenv import load_dotenv

# Reload the .env file and override any stale notebook environment values.
load_dotenv(override=True)

# Read the PostgreSQL host.
db_host = os.getenv("NETFLIX_DB_HOST")

# Read the PostgreSQL port.
db_port = os.getenv("NETFLIX_DB_PORT")

# Read the PostgreSQL database name.
db_name = os.getenv("NETFLIX_DB_NAME")

# Read the dedicated ETL account.
db_user = os.getenv("NETFLIX_ETL_USER")

# Read the ETL password without printing it.
db_password = os.getenv("NETFLIX_ETL_PASSWORD")

# Fail immediately if an important setting is missing.
required_values = {
    "NETFLIX_DB_HOST": db_host,
    "NETFLIX_DB_PORT": db_port,
    "NETFLIX_DB_NAME": db_name,
    "NETFLIX_ETL_USER": db_user,
    "NETFLIX_ETL_PASSWORD": db_password,
}

# Identify any missing variables.
missing_values = [
    name
    for name, value in required_values.items()
    if not value
]

# Stop execution before making a bad database connection.
if missing_values:
    raise ValueError(
        f"Missing environment variables: {missing_values}"
    )

connection = psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
)

with connection.cursor() as cursor:

    # Confirm the database and PostgreSQL role actually in use.
    cursor.execute(
        """
        SELECT
            current_database(),
            current_user;
        """
    )

    # Retrieve the returned row.
    database_name, database_user = cursor.fetchone()

# Display safe verification information.
print("Connection status: SUCCESS")
print("Database:", database_name)
print("Connected user:", database_user)

Connection status: SUCCESS
Database: postgres
Connected user: netflix_etl


## Initialise the pipeline

Create the first control record for the Netflix incremental pipeline.

This record acts as the pipeline's checkpoint and will later store the last successful watermark, last processed file and current pipeline status.

In [ ]:
# Import pandas 
import pandas as pd


# Give the pipeline a permanent identifier.
pipeline_name = "netflix_incremental_pipeline"


with connection.cursor() as cursor:

    # Create the pipeline control record if it does not already exist.
    cursor.execute(
        """
        INSERT INTO etl.pipeline_control (
            pipeline_name
        )
        VALUES (%s)
        ON CONFLICT (pipeline_name) DO NOTHING;
        """,
        (pipeline_name,),
    )


connection.commit()


# Read the control record back from PostgreSQL.
pipeline_state = pd.read_sql_query(
    """
    SELECT *
    FROM etl.pipeline_control
    WHERE pipeline_name = %s;
    """,
    connection,
    params=(pipeline_name,),
)


display(pipeline_state)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_46636/4206046250.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pipeline_state = pd.read_sql_query(


,pipeline_name,last_successful_load,last_watermark,last_file_name,rows_processed,status,updated_at
0,netflix_incremental_pipeline,2026-08-18 15:43:31.231873+00:00,2026-08-18 06:00:00+00:00,titles_20260818_0600.csv,250,FAILED,2026-08-18 16:14:07.465346+00:00


## Register an incoming batch

Simulate a new Netflix data file arriving and record it before processing begins.

The batch history table gives us an audit trail for every incoming file.

In [ ]:
# Create a fake filename representing an incoming batch.
test_file_name = "titles_20260818_0600.csv"


# Use a temporary checksum while testing the workflow.
# Later this value will be calculated automatically from the real file.
test_checksum = "TEST_CHECKSUM_001"


# Simulate how many records arrived in the batch.
test_rows_received = 250


# Open a PostgreSQL cursor for the batch insert.
with connection.cursor() as cursor:

    # Register the new batch before any transformation begins.
    cursor.execute(
        """
        INSERT INTO etl.batch_history (
            pipeline_name,
            file_name,
            file_checksum,
            rows_received,
            status
        )
        VALUES (%s, %s, %s, %s, 'RECEIVED')
        RETURNING batch_id;
        """,
        (
            pipeline_name,
            test_file_name,
            test_checksum,
            test_rows_received,
        ),
    )

    # Retrieve the unique batch ID generated by PostgreSQL.
    batch_id = cursor.fetchone()[0]


# Save the new batch record.
connection.commit()


print("Registered batch ID:", batch_id)

Registered batch ID: 5


In [6]:
# Retrieve the batch we just registered.
registered_batch = pd.read_sql_query(
    """
    SELECT *
    FROM etl.batch_history
    WHERE batch_id = %s;
    """,
    connection,
    params=(batch_id,),
)


# Display the registered batch.
display(registered_batch)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_46636/1773917316.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  registered_batch = pd.read_sql_query(


,batch_id,pipeline_name,file_name,file_checksum,received_at,processing_started_at,processing_completed_at,rows_received,rows_processed,status,error_message
0,5,netflix_incremental_pipeline,titles_20260818_0600.csv,TEST_CHECKSUM_001,2026-08-18 21:36:25.874170+00:00,None,None,250,None,RECEIVED,None


## Start processing

Change both the batch and the overall pipeline state to RUNNING.

In [ ]:
with connection.cursor() as cursor:

    # Mark the individual batch as actively processing.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'RUNNING',
            processing_started_at = CURRENT_TIMESTAMP
        WHERE batch_id = %s;
        """,
        (batch_id,),
    )

    # Mark the overall pipeline as actively running.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            status = 'RUNNING',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )


# Commit both updates.
connection.commit()

In [8]:
# Inspect the current states of both control tables.
current_pipeline_state = pd.read_sql_query(
    """
    SELECT *
    FROM etl.pipeline_control
    WHERE pipeline_name = %s;
    """,
    connection,
    params=(pipeline_name,),
)

current_batch_state = pd.read_sql_query(
    """
    SELECT *
    FROM etl.batch_history
    WHERE batch_id = %s;
    """,
    connection,
    params=(batch_id,),
)


# Display the current pipeline and batch states.
display(current_pipeline_state)
display(current_batch_state)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_46636/1977431691.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  current_pipeline_state = pd.read_sql_query(
/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_46636/1977431691.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  current_batch_state = pd.read_sql_query(


,pipeline_name,last_successful_load,last_watermark,last_file_name,rows_processed,status,updated_at
0,netflix_incremental_pipeline,2026-08-18 15:43:31.231873+00:00,2026-08-18 06:00:00+00:00,titles_20260818_0600.csv,250,RUNNING,2026-08-18 21:36:25.979321+00:00


,batch_id,pipeline_name,file_name,file_checksum,received_at,processing_started_at,processing_completed_at,rows_received,rows_processed,status,error_message
0,5,netflix_incremental_pipeline,titles_20260818_0600.csv,TEST_CHECKSUM_001,2026-08-18 21:36:25.874170+00:00,2026-08-18 21:36:25.979321+00:00,None,250,None,RUNNING,None


## Complete a successful batch

Simulate successful processing of the current batch.

The pipeline watermark must advance only after the batch reaches SUCCESS.

In [ ]:
# Import datetime so we can simulate the timestamp of the newest source record.
from datetime import datetime, timezone


# Simulate the newest source timestamp contained in this batch.
new_watermark = datetime(
    2026,
    8,
    18,
    6,
    0,
    tzinfo=timezone.utc,
)


# Simulate that every incoming row was processed successfully.
rows_processed = test_rows_received


# Open a PostgreSQL cursor so we can complete the batch.
with connection.cursor() as cursor:

    # Mark the individual batch as successfully completed.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'SUCCESS',
            rows_processed = %s,
            processing_completed_at = CURRENT_TIMESTAMP
        WHERE batch_id = %s;
        """,
        (
            rows_processed,
            batch_id,
        ),
    )

    # Advance the overall pipeline checkpoint only after the batch succeeds.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            last_successful_load = CURRENT_TIMESTAMP,
            last_watermark = %s,
            last_file_name = %s,
            rows_processed = %s,
            status = 'SUCCESS',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (
            new_watermark,
            test_file_name,
            rows_processed,
            pipeline_name,
        ),
    )


connection.commit()

In [ ]:
# Read the current pipeline control record.
successful_pipeline_state = pd.read_sql_query(
    """
    SELECT
        pipeline_name,
        last_successful_load,
        last_watermark,
        last_file_name,
        rows_processed,
        status
    FROM etl.pipeline_control
    WHERE pipeline_name = %s;
    """,
    connection,
    params=(pipeline_name,),
)


# Read the completed batch record.
successful_batch_state = pd.read_sql_query(
    """
    SELECT
        batch_id,
        file_name,
        received_at,
        processing_started_at,
        processing_completed_at,
        rows_received,
        rows_processed,
        status
    FROM etl.batch_history
    WHERE batch_id = %s;
    """,
    connection,
    params=(batch_id,),
)


display(successful_pipeline_state)

display(successful_batch_state)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_46636/2175619085.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  successful_pipeline_state = pd.read_sql_query(
/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_46636/2175619085.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  successful_batch_state = pd.read_sql_query(


,pipeline_name,last_successful_load,last_watermark,last_file_name,rows_processed,status
0,netflix_incremental_pipeline,2026-08-18 21:36:26.089254+00:00,2026-08-18 06:00:00+00:00,titles_20260818_0600.csv,250,SUCCESS


,batch_id,file_name,received_at,processing_started_at,processing_completed_at,rows_received,rows_processed,status
0,5,titles_20260818_0600.csv,2026-08-18 21:36:25.874170+00:00,2026-08-18 21:36:25.979321+00:00,2026-08-18 21:36:26.089254+00:00,250,250,SUCCESS


## Failed batch and watermark protection

Simulate a second incoming batch that fails during processing.

The previous successful watermark must remain unchanged.

In [ ]:
# Read the watermark before starting the failure test.

with connection.cursor() as cursor:

    cursor.execute(
        """
        SELECT last_watermark
        FROM etl.pipeline_control
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )

    # Store the watermark so we can compare it after the failure.
    watermark_before_failure = cursor.fetchone()[0]


print("Watermark before failure:", watermark_before_failure)

Watermark before failure: 2026-08-18 07:00:00+01:00


## Now register another batch

In [ ]:
# Create a second simulated incoming file.
failed_file_name = "titles_20260819_0600.csv"


# Give the second file a different test checksum.
failed_checksum = "TEST_CHECKSUM_002"


# Simulate the number of rows contained in the second batch.
failed_rows_received = 300


with connection.cursor() as cursor:

    cursor.execute(
        """
        INSERT INTO etl.batch_history (
            pipeline_name,
            file_name,
            file_checksum,
            rows_received,
            status
        )
        VALUES (%s, %s, %s, %s, 'RECEIVED')
        RETURNING batch_id;
        """,
        (
            pipeline_name,
            failed_file_name,
            failed_checksum,
            failed_rows_received,
        ),
    )

    # Capture the newly generated batch ID.
    failed_batch_id = cursor.fetchone()[0]


connection.commit()

print("Failed-test batch ID:", failed_batch_id)

Failed-test batch ID: 6


## Make it running

In [ ]:
with connection.cursor() as cursor:

    # Mark the test batch as RUNNING.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'RUNNING',
            processing_started_at = CURRENT_TIMESTAMP
        WHERE batch_id = %s;
        """,
        (failed_batch_id,),
    )

    # Mark the overall pipeline as RUNNING.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            status = 'RUNNING',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )


# Persist the RUNNING states.
connection.commit()

## Delibrately fail the simulation

In [ ]:
# Create a fake error message representing a transformation failure.
simulated_error = "Simulated transformation failure during incremental load."


# Open a cursor so the failed state can be recorded.
with connection.cursor() as cursor:

    # Mark the individual batch as FAILED.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'FAILED',
            processing_completed_at = CURRENT_TIMESTAMP,
            error_message = %s
        WHERE batch_id = %s;
        """,
        (
            simulated_error,
            failed_batch_id,
        ),
    )

    # Mark the overall pipeline as FAILED.
    # Notice that last_watermark is deliberately NOT updated.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            status = 'FAILED',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )


connection.commit()

## Perform critical check

In [ ]:
# Read the watermark after the simulated failure.
with connection.cursor() as cursor:

    cursor.execute(
        """
        SELECT
            last_watermark,
            status
        FROM etl.pipeline_control
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )

    # Store the returned values.
    watermark_after_failure, pipeline_status = cursor.fetchone()


# Display the comparison.
print("Watermark before failure:", watermark_before_failure)
print("Watermark after failure: ", watermark_after_failure)
print("Pipeline status:         ", pipeline_status)


# Confirm programmatically that the watermark did not move.
assert watermark_after_failure == watermark_before_failure


# Confirm that the pipeline correctly reports failure.
assert pipeline_status == "FAILED"

print("PASS: failed batch did not advance the watermark.")

Watermark before failure: 2026-08-18 07:00:00+01:00
Watermark after failure:  2026-08-18 07:00:00+01:00
Pipeline status:          FAILED
PASS: failed batch did not advance the watermark.


## Duplicate batch detection


Verify that an already successfully processed file can be identified before it enters the pipeline again.

In [ ]:
# Use the checksum from our already successful first batch.
incoming_checksum = test_checksum


# Open a cursor to check whether this source has already succeeded.
with connection.cursor() as cursor:

    cursor.execute(
        """
        SELECT
            batch_id,
            file_name,
            status
        FROM etl.batch_history
        WHERE file_checksum = %s
          AND status = 'SUCCESS'
        LIMIT 1;
        """,
        (incoming_checksum,),
    )

    # Retrieve the matching batch when one exists.
    duplicate_batch = cursor.fetchone()


# Determine whether the incoming file is a duplicate.
is_duplicate = duplicate_batch is not None


print("Duplicate detected:", is_duplicate)

print("Original batch:", duplicate_batch)


# Confirm that duplicate detection works.
assert is_duplicate is True

print("PASS: previously processed batch was detected.")

Duplicate detected: True
Original batch: (1, 'titles_20260818_0600.csv', 'SUCCESS')
PASS: previously processed batch was detected.


## Late-arriving records

Validate the incremental lookback strategy.

The pipeline intentionally re-reads a small period before the last successful watermark so records that arrive late are not missed.

In [ ]:
# Import timedelta so we can subtract a lookback period from the watermark.
from datetime import timedelta


with connection.cursor() as cursor:

    # Retrieve only the last successfully processed source timestamp.
    cursor.execute(
        """
        SELECT last_watermark
        FROM etl.pipeline_control
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )

    current_watermark = cursor.fetchone()[0]


# Define how far backwards each incremental load should re-read.
lookback_minutes = 10


# Create the true extraction starting point.
extract_from = current_watermark - timedelta(minutes=lookback_minutes)


# Display the calculated incremental window.
print("Current watermark:", current_watermark)
print("Lookback period:", lookback_minutes, "minutes")
print("Next extraction starts from:", extract_from)

Current watermark: 2026-08-18 07:00:00+01:00
Lookback period: 10 minutes
Next extraction starts from: 2026-08-18 06:50:00+01:00


## simulate a late record

In [18]:
# Simulate source records received during the next incremental batch.
simulated_records = [
    {"title_id": 101, "source_timestamp": current_watermark + timedelta(minutes=2)},
    {"title_id": 102, "source_timestamp": current_watermark + timedelta(minutes=5)},
    {"title_id": 103, "source_timestamp": current_watermark - timedelta(minutes=3)},
]


# Display the simulated records.
for record in simulated_records:
    print(record)

{'title_id': 101, 'source_timestamp': datetime.datetime(2026, 8, 18, 7, 2, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos'))}
{'title_id': 102, 'source_timestamp': datetime.datetime(2026, 8, 18, 7, 5, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos'))}
{'title_id': 103, 'source_timestamp': datetime.datetime(2026, 8, 18, 6, 57, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos'))}


## filter using our extraction boundary

In [19]:
# Keep every record that falls inside the incremental extraction window.
records_selected = [
    record
    for record in simulated_records
    if record["source_timestamp"] >= extract_from
]


# Display the records the next load would capture.
for record in records_selected:
    print(
        "Selected:",
        record["title_id"],
        record["source_timestamp"],
    )


# Confirm that the late 05:57 record was recovered.
selected_ids = [record["title_id"] for record in records_selected]


# The late-arriving record must be included.
assert 103 in selected_ids


# Display confirmation when the late record is successfully recovered.
print("PASS: late-arriving record was captured by the lookback window.")

Selected: 101 2026-08-18 07:02:00+01:00
Selected: 102 2026-08-18 07:05:00+01:00
Selected: 103 2026-08-18 06:57:00+01:00
PASS: late-arriving record was captured by the lookback window.


## Late batch arrival

Simulate a scheduled pipeline waiting for an expected source file rather than failing immediately when the file arrives slightly late

In [ ]:
# Import datetime utilities for the simulated schedule.
from datetime import datetime, timezone, timedelta


# Simulate the scheduled Airflow execution time.
scheduled_time = datetime(
    2026,
    8,
    18,
    9,
    0,
    tzinfo=timezone.utc,
)


# Simulate the source file arriving two minutes late.
actual_file_arrival = scheduled_time + timedelta(minutes=2)


# Define how long the pipeline is willing to wait.
allowed_wait_minutes = 10


# Calculate the latest acceptable arrival time.
arrival_deadline = scheduled_time + timedelta(minutes=allowed_wait_minutes)


# Check whether the source arrived within the permitted window.
file_arrived_within_sla = actual_file_arrival <= arrival_deadline


print("Scheduled run:", scheduled_time)
print("File arrival:", actual_file_arrival)
print("Deadline:", arrival_deadline)
print("Within SLA:", file_arrived_within_sla)


# Confirm that a file arriving two minutes late is still acceptable.
assert file_arrived_within_sla is True

print("PASS: slightly late batch would still be processed.")

Scheduled run: 2026-08-18 09:00:00+00:00
File arrival: 2026-08-18 09:02:00+00:00
Deadline: 2026-08-18 09:10:00+00:00
Within SLA: True
PASS: slightly late batch would still be processed.


## Test a genuinely missing batch

In [ ]:
# Simulate a file that arrives too late for the configured SLA.
very_late_file_arrival = scheduled_time + timedelta(minutes=30)


# Determine whether this arrival is still acceptable.
very_late_within_sla = very_late_file_arrival <= arrival_deadline

print("Very late arrival:", very_late_file_arrival)
print("Within SLA:", very_late_within_sla)


# Confirm that the pipeline would reject/timeout this arrival.
assert very_late_within_sla is False

print("PASS: excessively late batch would trigger the timeout path.")

Very late arrival: 2026-08-18 09:30:00+00:00
Within SLA: False
PASS: excessively late batch would trigger the timeout path.


## Production module test Database connection

### Confirm that the reusable production database module connects with the same `netflix_etl` identity validated during notebook development.

In [ ]:
import sys

from pathlib import Path


# Find the project root from the current notebook location.
project_root = Path.cwd().parent


if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


from src.database import get_etl_connection


production_connection = get_etl_connection()


# Open a cursor to verify the database identity.
with production_connection.cursor() as cursor:

    cursor.execute(
        """
        SELECT
            current_database(),
            current_user;
        """
    )

    # Retrieve the returned values.
    database_name, connected_user = cursor.fetchone()


print("Database:", database_name)
print("Connected user:", connected_user)


# Prove that production code uses the intended ETL account.
assert connected_user == "netflix_etl"


print("PASS: production database module connected successfully.")

Database: postgres
Connected user: netflix_etl
PASS: production database module connected successfully.


## close production test

In [23]:
# Close the production test connection when testing is complete.
production_connection.close()

# Confirm that the test connection has been cleaned up.
print("Production test connection closed.")

Production test connection closed.


## Production module test: Duplicate detection

Verify that the reusable batch tracker identifies the successful batch already created during simulation.

In [ ]:
# Import the production duplicate-detection function.
from src.batch_tracker import find_successful_batch_by_checksum


# Search using the checksum from our successful test batch.
existing_batch = find_successful_batch_by_checksum(
    "TEST_CHECKSUM_001"
)


print("Existing batch:", existing_batch)


# Confirm that production duplicate detection found the batch.
assert existing_batch is not None


# Confirm that PostgreSQL reports the original batch as successful.
assert existing_batch[-1] == "SUCCESS"


print("PASS: production duplicate detection works.")

Existing batch: (1, 'netflix_incremental_pipeline', 'titles_20260818_0600.csv', 'TEST_CHECKSUM_001', 'SUCCESS')
PASS: production duplicate detection works.


## Test registration using production code

In [ ]:
# Import the reusable batch-registration function.
from src.batch_tracker import register_batch


# Register a new production-style test batch.
production_test_batch_id = register_batch(
    pipeline_name="netflix_incremental_pipeline",
    file_name="production_module_test.csv",
    file_checksum="PRODUCTION_TEST_CHECKSUM_001",
    rows_received=100,
)


# Display the generated identifier.
print("Production test batch ID:", production_test_batch_id)


# Confirm PostgreSQL generated a valid identifier.
assert production_test_batch_id is not None

print("PASS: production batch registration works.")

Production test batch ID: 7
PASS: production batch registration works.


## Production module test: Watermark logic

Validate the production watermark module using the same pipeline state already created during incremental-load testing.

In [ ]:
from src.watermark import (
    get_last_watermark,
    get_extraction_start,
)


# Read the currently stored successful watermark.
production_watermark = get_last_watermark(
    "netflix_incremental_pipeline"
)


# Calculate the next extraction start using a 10-minute lookback.
production_extraction_start = get_extraction_start(
    "netflix_incremental_pipeline",
    lookback_minutes=10,
)


print("Stored watermark:", production_watermark)


# Display the calculated incremental extraction boundary.
print(
    "Extraction starts from:",
    production_extraction_start,
)


# Confirm that both values exist.
assert production_watermark is not None
assert production_extraction_start is not None


# Confirm the extraction boundary occurs before the watermark.
assert production_extraction_start < production_watermark

print("PASS: production watermark logic works.")

Stored watermark: 2026-08-18 07:00:00+01:00
Extraction starts from: 2026-08-18 06:50:00+01:00
PASS: production watermark logic works.


## Production module test: File checksum

Validate that the production checksum utility generates a stable SHA-256 fingerprint for the same source file.

In [ ]:
from pathlib import Path

# Import the production checksum utility.
from src.file_utils import calculate_file_checksum


# Build the path to the real Netflix titles source file.
titles_file = project_root / "data" / "raw" / "titles.csv"


assert titles_file.exists(), f"File not found: {titles_file}"


# Calculate the checksum for the first time.
checksum_1 = calculate_file_checksum(titles_file)


# Calculate the checksum again without changing the file.
checksum_2 = calculate_file_checksum(titles_file)


# Display the generated SHA-256 fingerprint.
print("Titles checksum:", checksum_1)


# Confirm that the checksum has the expected SHA-256 length.
assert len(checksum_1) == 64


# Confirm that the same file always generates the same checksum.
assert checksum_1 == checksum_2


# Display confirmation when all checksum tests pass.
print("PASS: checksum generation is deterministic.")

Titles checksum: 639cba13a200033e1ebb8e71243eef2d6d4453706ccff67e101f4935dbaad012
PASS: checksum generation is deterministic.


## Prove that changing the content changes the checksum

In [ ]:
# Create a temporary test file inside the project data directory.
temporary_file = project_root / "data" / "checksum_test.txt"


# Write the first version of the file.
temporary_file.write_text(
    "Netflix batch version 1",
    encoding="utf-8",
)


# Calculate the checksum of version 1.
checksum_before_change = calculate_file_checksum(temporary_file)


# Change the file contents.
temporary_file.write_text(
    "Netflix batch version 2",
    encoding="utf-8",
)


# Calculate the checksum again after modifying the content.
checksum_after_change = calculate_file_checksum(temporary_file)


print("Before:", checksum_before_change)
print("After: ", checksum_after_change)


# Different contents should create different checksums.
assert checksum_before_change != checksum_after_change


# Remove the temporary file after the test.
temporary_file.unlink()


# Confirm that checksum changes when file content changes.
print("PASS: changed content produces a different checksum.")

Before: e8692fa8643ac7592dcaef074522e98f58f148b027bf1eeb914bf93fc2cff201
After:  de62e22d0590e7d53efdd097871cb425fe9d733e0c49d0f1ac6f44a0278b8466
PASS: changed content produces a different checksum.


## Connect checksum + duplication detection

In [31]:
# Import the production duplicate-detection function.
from src.batch_tracker import find_successful_batch_by_checksum


# Calculate the checksum of the real source file.
real_titles_checksum = calculate_file_checksum(titles_file)


# Search batch history for the same successfully processed content.
existing_real_batch = find_successful_batch_by_checksum(
    real_titles_checksum
)


# Display the result.
print("Existing successful batch:", existing_real_batch)

Existing successful batch: None
